# 🏥 Fine-Tuning de LaMa (Large Mask Inpainting) en Google Colab para Ecografía Mamaria

Este cuaderno ejecuta el **reentrenamiento autosupervisado** del modelo `big-lama` en Google Colab con GPU (**NVIDIA T4 o A100**), utilizando tu dataset curado de **765 ecografías limpias**.

### 🎯 Objetivos Clave:
1. **Adaptar los filtros de Fourier (FFC)** desde fotografía óptica (*Places2*) hacia la física del ultrasonido (*speckle* acústico y atenuación tisular).
2. **Entrenar con generador sintético especializado** en calipers cruciformes (`+`, `×`), líneas de medición y texto médico (8–14 px).
3. **Preservar la transición en bordes tumorales** (criterios BI-RADS) sin desvanecer los márgenes lesionales.

## ⚙️ Paso 1: Verificar GPU Asignada en Colab

In [ ]:
# Verificar que la GPU esté activa (Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU)
!nvidia-smi

## 📁 Paso 2: Conectar Google Drive (Persistencia de Checkpoints y Dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/LaMa_Ultrasound_Project"
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print(f"Directorio de trabajo en Drive listo: {DRIVE_PROJECT_DIR}")

## 📥 Paso 3: Clonar el Repositorio Oficial de LaMa e Instalar Dependencias

In [ ]:
%cd /content
!git clone https://github.com/advimman/lama.git
%cd /content/lama

# Instalación de dependencias del framework LaMa
!pip install -q albumentations==1.3.1 hydra-core==1.1.0 pytorch-lightning==1.2.9 kornia==0.5.0 webdataset==0.1.40 easydict pyyaml

## 📦 Paso 4: Descargar el Checkpoint Oficial Preentrenado `big-lama`

In [ ]:
%cd /content/lama
import os

os.makedirs('/content/lama/models', exist_ok=True)

if not os.path.exists('/content/lama/models/big-lama'):
    print('Descargando checkpoint preentrenado oficial big-lama desde Hugging Face...')
    !curl -L -o /content/lama/models/big-lama.zip https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.zip
    print('Descomprimiendo modelo...')
    !unzip -q -o /content/lama/models/big-lama.zip -d /content/lama/models/
    print('✓ Checkpoint listo en /content/lama/models/big-lama')
else:
    print('✓ El modelo big-lama ya se encuentra listo.')

## 🗂️ Paso 5: Descomprimir tu Dataset de Ecografías Limpias (`ecografias_limpias_para_drive.zip`)

Busca el archivo ZIP subido a tu Google Drive y lo descomprime en el almacenamiento rápido local de Colab, organizándolo en `train/` y `val/`.

In [ ]:
import zipfile
import shutil
from pathlib import Path

# Ruta al zip en tu Google Drive
zip_drive_path = "/content/drive/MyDrive/LaMa_Ultrasound_Project/ecografias_limpias_para_drive.zip"

# Si lo subiste directamente a la raíz de Colab, alternativa:
if not os.path.exists(zip_drive_path) and os.path.exists("/content/ecografias_limpias_para_drive.zip"):
    zip_drive_path = "/content/ecografias_limpias_para_drive.zip"

print(f"Extrayendo dataset desde: {zip_drive_path}...")
raw_extract_dir = Path("/content/dataset_raw")
raw_extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_drive_path, 'r') as zf:
    zf.extractall(raw_extract_dir)

# Encontrar todas las imágenes descomprimidas
images = list(raw_extract_dir.rglob("*.png")) + list(raw_extract_dir.rglob("*.jpg"))
print(f"✓ Total de ecografías limpias extraídas: {len(images)}")

# Estructurar automáticamente en train (85%) y val (15%)
train_dir = Path("/content/dataset/train")
val_dir = Path("/content/dataset/val")
train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

import random
random.seed(42)
random.shuffle(images)

val_count = max(10, int(len(images) * 0.15))
val_images = set(images[:val_count])

for img_p in images:
    dest = val_dir if img_p in val_images else train_dir
    shutil.copy2(img_p, dest / img_p.name)

print(f"✓ Partición completada: Train = {len(list(train_dir.glob('*')))} | Val = {len(list(val_dir.glob('*')))}")

## 🩺 Paso 6: Generador Especializado de Máscaras Sintéticas (Calipers y Texto)

Genera dinámicamente en el Dataloader:
- Cruces ortogonales (`+`) y oblicuas (`×`) de 1–3 px.
- Trazos de medición punteados.
- Cadenas de texto médico (8–14 px) ubicadas preferentemente sobre el parénquima y zonas hipoecoicas.

In [ ]:
import numpy as np
import cv2
import random
from typing import Tuple, Optional

class UltrasoundArtifactMaskGenerator:
    def __init__(
        self,
        img_size: Tuple[int, int] = (512, 512),
        min_calipers: int = 2,
        max_calipers: int = 5,
        line_thickness_range: Tuple[int, int] = (1, 3),
        caliper_arm_length_range: Tuple[int, int] = (12, 26),
        include_text: bool = True,
        include_dotted_lines: bool = True,
        nodule_bias_probability: float = 0.65,
    ):
        self.height, self.width = img_size
        self.min_calipers = min_calipers
        self.max_calipers = max_calipers
        self.thickness_range = line_thickness_range
        self.arm_range = caliper_arm_length_range
        self.include_text = include_text
        self.include_dotted_lines = include_dotted_lines
        self.nodule_bias_prob = nodule_bias_probability
        self.sample_texts = ["D1: {d1:.2f}cm", "D2: {d2:.2f}cm", "{d1:.1f}x{d2:.1f}mm", "7.5MHz", "12MHz", "DR: 65", "MI: 0.8", "RAD 9:00"]

    def _get_target_location(self, image: Optional[np.ndarray] = None) -> Tuple[int, int]:
        margin = 40
        if image is not None and random.random() < self.nodule_bias_prob:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
            h, w = gray.shape
            roi = gray[margin : h - margin, margin : w - margin]
            candidate_y, candidate_x = np.where((roi > 15) & (roi < 90))
            if len(candidate_x) > 0:
                idx = random.randint(0, len(candidate_x) - 1)
                return candidate_x[idx] + margin, candidate_y[idx] + margin
        return random.randint(margin, self.width - margin), random.randint(margin, self.height - margin)

    def generate(self, image: Optional[np.ndarray] = None) -> np.ndarray:
        mask = np.zeros((self.height, self.width), dtype=np.uint8)
        num_calipers = random.randint(self.min_calipers, self.max_calipers)
        pts = []
        for _ in range(num_calipers):
            pt = self._get_target_location(image)
            pts.append(pt)
            arm = random.randint(*self.arm_range)
            t = random.randint(*self.thickness_range)
            if random.random() < 0.65:
                cv2.line(mask, (pt[0] - arm, pt[1]), (pt[0] + arm, pt[1]), 255, t)
                cv2.line(mask, (pt[0], pt[1] - arm), (pt[0], pt[1] + arm), 255, t)
            else:
                d = int(arm * 0.707)
                cv2.line(mask, (pt[0] - d, pt[1] - d), (pt[0] + d, pt[1] + d), 255, t)
                cv2.line(mask, (pt[0] - d, pt[1] + d), (pt[0] + d, pt[1] - d), 255, t)

        if self.include_dotted_lines and len(pts) >= 2:
            p1, p2 = pts[0], pts[1]
            dist = np.hypot(p2[0] - p1[0], p2[1] - p1[1])
            steps = max(2, int(dist / 6))
            for i in range(1, steps):
                r = i / float(steps)
                cv2.circle(mask, (int(p1[0]*(1-r) + p2[0]*r), int(p1[1]*(1-r) + p2[1]*r)), 1, 255, -1)

        if self.include_text:
            pos = self._get_target_location(image)
            txt = random.choice(self.sample_texts).format(d1=random.uniform(0.8, 3.0), d2=random.uniform(0.5, 2.0))
            cv2.putText(mask, txt, (max(10, min(pos[0], self.width-120)), max(20, min(pos[1], self.height-20))), cv2.FONT_HERSHEY_SIMPLEX, 0.4, 255, 1, cv2.LINE_AA)
        return mask

print("✓ Generador sintético de artefactos ecográficos cargado con éxito.")

## 🖼️ Paso 7: Visualizar la Simulación de Entrenamiento sobre tus Ecografías

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Tomar la primera ecografía real de tu dataset limpio
sample_path = list(train_dir.glob("*.png"))[0]
real_us = cv2.imread(str(sample_path))
real_us = cv2.resize(real_us, (512, 512))

gen = UltrasoundArtifactMaskGenerator((512, 512))
mask = gen.generate(real_us)

# Corromper con la máscara: X = Y * (1 - M)
corrupted = real_us.copy()
corrupted[mask == 255] = 255

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv2.cvtColor(real_us, cv2.COLOR_BGR2RGB))
ax[0].set_title(f"Ground Truth Y\n({sample_path.name})")
ax[0].axis('off')

ax[1].imshow(mask, cmap='gray')
ax[1].set_title("Máscara M Generada Dinámicamente\n(Calipers + Texto Clínico)")
ax[1].axis('off')

ax[2].imshow(cv2.cvtColor(corrupted, cv2.COLOR_BGR2RGB))
ax[2].set_title("Entrada X Corrupta\n[X = Y ⊙ (1 - M)]")
ax[2].axis('off')
plt.tight_layout()
plt.show()

## 🚀 Paso 8: Copiar Configuración Hydra y Lanzar Fine-Tuning

Este comando ejecuta el fine-tuning utilizando:
- Checkpoint base: `big-lama`
- Optimizador: AdamW con tasa de aprendizaje baja ($10^{-4}$)
- Discriminador adversarial activo (para preservar la granularidad del speckle)
- Checkpoints guardados en `/content/drive/MyDrive/LaMa_Ultrasound_Project/checkpoints`

In [ ]:
# ==============================================================================
# PASO 8: INICIALIZACIÓN COMPLETA, MÁSCARAS DE VALIDACIÓN Y LANZAMIENTO
# ==============================================================================

# 1. Posicionarse en la raíz segura
%cd /content
import os, sys, shutil, glob, random, cv2
import numpy as np
from pathlib import Path

# 2. Configurar variables de entorno esenciales
os.environ['TORCH_HOME'] = '/content/lama'
os.environ['PYTHONPATH'] = '/content/lama'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# 3. Preparar pesos de ADE20K para la pérdida perceptual ResNetPL (~95 MB)
ade_dir = '/content/lama/ade20k/ade20k-resnet50dilated-ppm_deepsup'
drive_ade_dir = '/content/drive/MyDrive/LaMa_Ultrasound_Project/models/ade20k/ade20k-resnet50dilated-ppm_deepsup'
os.makedirs(ade_dir, exist_ok=True)
os.makedirs(drive_ade_dir, exist_ok=True)

encoder_path = os.path.join(ade_dir, 'encoder_epoch_20.pth')
drive_encoder = os.path.join(drive_ade_dir, 'encoder_epoch_20.pth')

if os.path.exists(drive_encoder) and os.path.getsize(drive_encoder) > 50000000:
    print('Recuperando pesos ADE20K desde Google Drive...')
    shutil.copy2(drive_encoder, encoder_path)
    print('✓ Pesos ADE20K listos desde Drive.')
elif not os.path.exists(encoder_path) or os.path.getsize(encoder_path) < 50000000:
    print('Descargando pesos ADE20K ResNet-50 (encoder_epoch_20.pth, ~95 MB)...')
    !curl -L -o {encoder_path} http://sceneparsing.csail.mit.edu/model/pytorch/ade20k-resnet50dilated-ppm_deepsup/encoder_epoch_20.pth
    if os.path.exists(encoder_path) and os.path.getsize(encoder_path) > 50000000:
        try:
            shutil.copy2(encoder_path, drive_encoder)
            print('✓ Pesos ADE20K respaldados en Google Drive.')
        except Exception:
            pass

# 4. Preparar directorios y máscaras fijas para validación (val y visual_test)
val_dir = Path('/content/dataset/val')
vis_dir = Path('/content/dataset/visual_test')
vis_dir.mkdir(parents=True, exist_ok=True)

# Obtener imágenes limpias de validación
val_imgs = [p for p in val_dir.glob('*.png') if not p.stem.endswith('_mask')] + \
           [p for p in val_dir.glob('*.jpg') if not p.stem.endswith('_mask')]

# Poblar visual_test con 10 muestras si está vacío
if len(list(vis_dir.glob('*'))) == 0 and len(val_imgs) > 0:
    for img_p in val_imgs[:10]:
        shutil.copy2(img_p, vis_dir / img_p.name)

# Generador sintético de calipers y texto para las máscaras de evaluación
def crear_mascara_ecografia(img_shape):
    h, w = img_shape[:2]
    m = np.zeros((h, w), dtype=np.uint8)
    for _ in range(random.randint(2, 4)):
        cx, cy = random.randint(30, max(31, w-30)), random.randint(30, max(31, h-30))
        arm, t = random.randint(12, 24), random.randint(1, 2)
        if random.random() < 0.6:
            cv2.line(m, (cx - arm, cy), (cx + arm, cy), 255, t)
            cv2.line(m, (cx, cy - arm), (cx, cy + arm), 255, t)
        else:
            d = int(arm * 0.707)
            cv2.line(m, (cx - d, cy - d), (cx + d, cy + d), 255, t)
            cv2.line(m, (cx - d, cy + d), (cx + d, cy - d), 255, t)
    txt = random.choice(['D1: 1.45cm', 'D2: 0.82cm', '7.5MHz', 'MI: 0.8'])
    cv2.putText(m, txt, (random.randint(10, max(10, w-120)), random.randint(20, max(20, h-20))), cv2.FONT_HERSHEY_SIMPLEX, 0.45, 255, 1, cv2.LINE_AA)
    return m

print('Sanitizando nombres y preparando pares imagen-máscara...')
# 1. Eliminar máscaras residuales anteriores
for folder in [train_dir, val_dir, vis_dir]:
    for mf in list(folder.glob('*_mask.png')) + list(folder.glob('*mask*.png')):
        try:
            mf.unlink()
        except Exception:
            pass

# 2. Renombrar imágenes para que NINGUNA contenga la palabra 'mask' en su nombre base
for folder in [train_dir, val_dir, vis_dir]:
    for img_p in list(folder.glob('*.png')) + list(folder.glob('*.jpg')):
        if 'mask' in img_p.name.lower():
            clean_name = img_p.name.replace('and_masks_', 'img_').replace('masks_', 'img_').replace('mask', 'img')
            img_p.rename(folder / clean_name)

def resize_and_pad(img, target_size=(256, 256)):
    h, w = img.shape[:2]
    th, tw = target_size
    scale = min(th / h, tw / w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    top = (th - nh) // 2
    bottom = th - nh - top
    left = (tw - nw) // 2
    right = tw - nw - left
    if len(img.shape) == 3:
        padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[0, 0, 0])
    else:
        padded = cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=0)
    return padded

# 3. Estandarizar resolución (256x256) y generar máscaras fijas (_mask.png) para val y visual_test
for folder in [val_dir, vis_dir]:
    imgs = [p for p in folder.glob('*.png') if not p.stem.endswith('_mask')] + \
           [p for p in folder.glob('*.jpg') if not p.stem.endswith('_mask')]
    for img_p in imgs:
        im = cv2.imread(str(img_p))
        if im is not None:
            # Homogeneizar dimensiones para que DataLoader pueda apilar batches sin error
            im_std = resize_and_pad(im, (256, 256))
            cv2.imwrite(str(img_p), im_std)
            mask_path = folder / f"{img_p.stem}_mask.png"
            mask = crear_mascara_ecografia(im_std.shape)
            cv2.imwrite(str(mask_path), mask)

print(f"✓ Máscaras listas: {len(list(val_dir.glob('*_mask.png')))} en val, {len(list(vis_dir.glob('*_mask.png')))} en visual_test.")

# 4. Parche preventivo para saicinpainting/evaluation/data.py
eval_data_path = '/content/lama/saicinpainting/evaluation/data.py'
if os.path.exists(eval_data_path):
    with open(eval_data_path, 'r') as f:
        ed_code = f.read()
    old_eval_glob = "self.mask_filenames = sorted(list(glob.glob(os.path.join(self.datadir, '**', '*mask*.png'), recursive=True)))"
    if old_eval_glob in ed_code:
        new_eval_glob = "all_m = sorted(list(glob.glob(os.path.join(self.datadir, '**', '*_mask*.png'), recursive=True)))\n        self.mask_filenames = [m for m in all_m if os.path.isfile(m.rsplit('_mask', 1)[0] + img_suffix)]"
        ed_code = ed_code.replace(old_eval_glob, new_eval_glob)
        with open(eval_data_path, 'w') as f:
            f.write(ed_code)
        print('✓ saicinpainting/evaluation/data.py blindado para pares de validación.')

# 5. Parche preventivo para saicinpainting/training/data/datasets.py (soporte PNG en train)
train_ds_path = '/content/lama/saicinpainting/training/data/datasets.py'
if os.path.exists(train_ds_path):
    with open(train_ds_path, 'r') as f:
        td_code = f.read()
    old_train_glob = "self.in_files = list(glob.glob(os.path.join(indir, '**', '*.jpg'), recursive=True))"
    if old_train_glob in td_code:
        new_train_glob = "all_train = list(glob.glob(os.path.join(indir, '**', '*.png'), recursive=True)) + list(glob.glob(os.path.join(indir, '**', '*.jpg'), recursive=True))\n        self.in_files = [f for f in all_train if not f.endswith('_mask.png')]"
        td_code = td_code.replace(old_train_glob, new_train_glob)
        with open(train_ds_path, 'w') as f:
            f.write(td_code)
        print('✓ saicinpainting/training/data/datasets.py actualizado para PNG en train.')

# 5. Parche preventivo para Kornia en fake_fakes.py
ff_path = '/content/lama/saicinpainting/training/modules/fake_fakes.py'
if os.path.exists(ff_path):
    with open(ff_path, 'r') as f:
        ff_code = f.read()
    if 'from kornia import SamplePadding' in ff_code:
        ff_code = ff_code.replace('from kornia import SamplePadding', 'from kornia.constants import SamplePadding')
        with open(ff_path, 'w') as f:
            f.write(ff_code)
        print('✓ Parche Kornia SamplePadding aplicado.')

# 6. Escribir bin/train.py limpio con compatibilidad Python 3.13, OmegaConf, NumPy 2.x y Transfer Learning
train_py_code = '''#!/usr/bin/env python3
# --- Parches de compatibilidad para Python 3.13 + OmegaConf + NumPy 2.x ---
import sys, os, typing, logging, traceback

sys.modules['typing.io'] = typing
sys.modules['typing.re'] = typing

try:
    from omegaconf import OmegaConf
    OmegaConf.register_new_resolver('env', lambda *args: os.environ.get(args[0], args[1] if len(args) > 1 else ''), replace=True)
except Exception:
    pass

try:
    import numpy as np
    np.Inf = np.inf
    np.Infinity = np.inf
    np.NaN = np.nan
    np.NAN = np.nan
    if not hasattr(np, 'sctypes'):
        np.sctypes = {
            'int': [np.int8, np.int16, np.int32, np.int64],
            'uint': [np.uint8, np.uint16, np.uint32, np.uint64],
            'float': [np.float16, np.float32, np.float64],
            'complex': [np.complex64, np.complex128],
            'others': [bool, object, bytes, str, np.void]
        }
except Exception:
    pass

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

import hydra
from omegaconf import OmegaConf
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.plugins import DDPPlugin

from saicinpainting.training.trainers import make_training_model
from saicinpainting.utils import register_debug_signal_handlers, handle_ddp_subprocess, handle_ddp_parent_process, \
    handle_deterministic_config

LOGGER = logging.getLogger(__name__)

@handle_ddp_subprocess()
@hydra.main(config_path='../configs/training', config_name='tiny_test.yaml')
def main(config: OmegaConf):
    try:
        need_set_deterministic = handle_deterministic_config(config)
        if sys.platform != 'win32':
            register_debug_signal_handlers()
        is_in_ddp_subprocess = handle_ddp_parent_process()
        config.visualizer.outdir = os.path.join(os.getcwd(), config.visualizer.outdir)
        if not is_in_ddp_subprocess:
            LOGGER.info(OmegaConf.to_yaml(config))
            OmegaConf.save(config, os.path.join(os.getcwd(), 'config.yaml'))
        checkpoints_dir = os.path.join(os.getcwd(), 'models')
        os.makedirs(checkpoints_dir, exist_ok=True)
        metrics_logger = TensorBoardLogger(config.location.tb_dir, name=os.path.basename(os.getcwd()))
        metrics_logger.log_hyperparams(config)
        training_model = make_training_model(config)
        # Transfer Learning: cargar pesos preentrenados del generador big-lama (PyTorch 2.6 weights_only=False)
        base_ckpt = '/content/lama/models/big-lama/models/best.ckpt'
        if os.path.exists(base_ckpt):
            try:
                import torch
                LOGGER.info(f'Cargando pesos preentrenados de {base_ckpt}...')
                raw_s = torch.load(base_ckpt, map_location="cpu", weights_only=False)
                s = raw_s.get("state_dict", raw_s)
                gen_s = {k.replace("generator.", ""): v for k, v in s.items() if k.startswith("generator.")}
                if not gen_s:
                    gen_s = s
                training_model.generator.load_state_dict(gen_s, strict=False)
                LOGGER.info('✓ Pesos del generador Big-LaMa transferidos con éxito para Fine-Tuning.')
            except Exception as e:
                LOGGER.warning(f'Nota transferencia de pesos: {e}')
        trainer_kwargs = OmegaConf.to_container(config.trainer.kwargs, resolve=True)
        if need_set_deterministic:
            trainer_kwargs['deterministic'] = True
        # Ajuste dinamico para datasets medicos (< 25000 batches)
        trainer_kwargs['limit_train_batches'] = 1.0
        trainer_kwargs['val_check_interval'] = 1.0
        trainer_kwargs['log_every_n_steps'] = 10
        trainer = Trainer(
            callbacks=ModelCheckpoint(dirpath=checkpoints_dir, **config.trainer.checkpoint_kwargs),
            logger=metrics_logger,
            default_root_dir=os.getcwd(),
            **trainer_kwargs
        )
        trainer.fit(training_model)
    except KeyboardInterrupt:
        LOGGER.warning('Interrupted by user')
    except Exception as ex:
        LOGGER.critical(f'Training failed due to {ex}:\\n{traceback.format_exc()}')
        sys.exit(1)

if __name__ == '__main__':
    main()
'''
with open('/content/lama/bin/train.py', 'w') as f:
    f.write(train_py_code)
print('✓ bin/train.py configurado limpiamente con parches y Transfer Learning.')

# 7. Actualizar aug.py para Albumentations moderno
aug_path = '/content/lama/saicinpainting/training/data/aug.py'
safe_aug = '''# Compatible con Albumentations moderno
import albumentations as A
try:
    from albumentations.core.transforms_interface import BasicTransform
except ImportError:
    from albumentations import BasicTransform

class IAAAffine2(BasicTransform):
    def __init__(self, scale=(0.7, 1.3), rotate=0.0, shear=(-0.1, 0.1), always_apply=False, p=0.5, **kwargs):
        super().__init__(p=p)
        self.affine = A.Affine(scale=scale, rotate=rotate, shear=shear, p=1.0)
    @property
    def targets(self):
        return {'image': self.apply, 'mask': self.apply_to_mask}
    def apply(self, img, **params):
        return self.affine(image=img)['image']
    def apply_to_mask(self, mask, **params):
        return self.affine(image=mask)['image']

class IAAPerspective2(BasicTransform):
    def __init__(self, scale=(0.0, 0.06), always_apply=False, p=0.5, **kwargs):
        super().__init__(p=p)
        self.persp = A.Perspective(scale=scale, p=1.0)
    @property
    def targets(self):
        return {'image': self.apply, 'mask': self.apply_to_mask}
    def apply(self, img, **params):
        return self.persp(image=img)['image']
    def apply_to_mask(self, mask, **params):
        return self.persp(image=mask)['image']
'''
if os.path.exists('/content/lama/saicinpainting/training/data'):
    with open(aug_path, 'w') as f:
        f.write(safe_aug)
    print('✓ aug.py actualizado.')

# 8. Configurar ubicación colab.yaml
os.makedirs('/content/lama/configs/training/location', exist_ok=True)
with open('/content/lama/configs/training/location/colab.yaml', 'w') as f:
    f.write('data_root_dir: /content/dataset\n'
            'out_root_dir: /content/drive/MyDrive/LaMa_Ultrasound_Project/experiments\n'
            'tb_dir: /content/drive/MyDrive/LaMa_Ultrasound_Project/tb_logs\n'
            'pretrained_models_dir: /content/lama/models\n')

# 9. Iniciar fine-tuning en GPU T4 con persistencia en Google Drive
%cd /content/lama
!TORCH_HOME=/content/lama PYTHONPATH=/content/lama python bin/train.py \
  -cn=big-lama \
  location=colab \
  data.train.indir=/content/dataset/train \
  data.val.indir=/content/dataset/val \
  data.visual_test.indir=/content/dataset/visual_test \
  hydra.run.dir=/content/drive/MyDrive/LaMa_Ultrasound_Project/experiments \
  data.val_batch_size=1 \
  trainer.kwargs.gpus=1 \
  trainer.kwargs.max_epochs=35
